In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 알고리즘을 사용하기 위해 import 한다.

선형 회귀 분석은 예측 문제를 풀기에는 적합하지만 분류 문제를 풀기에는 적합하지 않다.

선형 회귀 분석은 레이블 값의 범위에 제한이 없어서 결과가 제한되는 상황에서 회귀 모형이 결과 값에 제한이 없다면 분류 문제를 풀기 어려워진다. 0과 1로 분류해야 하는 문제에서 모델의 결과값은 오직 0과 1 사이의 값으로 나와야 할 것이다. 이러한 문제점을 해결하기 위해 사용하는 방법이 로지스틱 회귀 분석이다.

$$z = w^Tx + b$$

위의 식은 선형 회귀 분석 모델이다.  
$z$값은 제한이 없고 어떤 값도 가질 수 있으므로 위 식을 이용해서 분류 문제를 푸는 것은 어렵다는 것을 알 수 있다. 기존의 선형 회귀 모델 식을 분류 문제를 풀 수 있도록 변형시키는 과정이 필요하다. 이를 위해 결과값이 제한된 범위를 가지도록 변형시켜 보자.

$$y = \frac {1} {1 + e^{-z}} = \frac {1} {1 + e^{-(w^Tx + b)}}$$

$z$에 대한  선형 회귀 식을 위와 같이 변경시키면 새로운 출력 $y$는 0과 1 사이의 값만 가지게 된다. 위의 함수를 시그모이드(sigmoid) 함수라고 부른다. 시그모이드 함수는 딥러닝에서도 자주 나오는 함수로 식의 우변이 $z = w^Tx + b$ 형태가 되도록 변경시키면 아래와 같이 표현할 수 있다.

$$log(\frac {y} {1 - y}) = w^Tx + b$$

위의 식을 보면 우변이 $z = w^Tx + b$와 같은 선형 형태로 나타난다. 이때, 좌변의 $\frac {y} {1 - y}$를 오즈 비(odds ratio)라고 부른다.

오즈 비에서 분자에 해당하는 $y$가 사건이 발생할 확률(성공 확률)이라고 했을 때 분모인 $1 - y$는 사건이 발생하지 않을 확률(실패 확률)에 해당된다. 실패 확률과 성공 확률의 비를 오즈 비라고 부른다. 또한 오즈 비에 log를 취한값 $log(\frac {y} {1 - y})$을 로짓(logit)이라고 부른다.

위스콘신 암 데이터를 사용해서 양성, 악성을 분류하는 모델을 생성하고 학습시킨다.

In [2]:
# 데이터 불러오기
raw_data = datasets.load_breast_cancer() # 사이킷런 라이브러리가 제공하는 위스콘신 암 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
# print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
# print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

# 모델 생성 후 데이터 학습
# 로지스틱 회귀 모델은 penalty 속성으로 제약 방식을 지정해서 모델을 만드는데 sklearn 1.10.0 버전 부터는 완전히 제거되어 사용할 수 없게된다.
# sklearn 1.10.0 버전 부터는 penalty 대신 l1_ratio 속성과 C 속성을 조합하여 규제의 종류를 설정해서 모델을 만들어야 한다.
# l1_ratio 속성에 0을 사용하면 릿지(L2 규제)를 사용하고 1을 사용하면 라쏘(L1 규제)를 사용한다. 비율을 사용하면 L1 + L2 혼합 규제를 사용한다.
# C는 규제 강도의 역수로, 값이 작을수록 규제가 강해지고, 값이 클수록 규제가 약해진다.
# L1, L2 제약을 모두 지원할 수 있도록 solver 속성값을 'saga'로 지정한다.
# model = LogisticRegression(penalty='l1', solver='saga') # 버전 1.9 까지 로지스틱 회귀 모델을 만든다.
# model = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga') # 버전 1.10 까지 로지스틱 회귀 모델을 만든다.
# model.fit(x_train, y_train) # 표준화된 학습 데이터(x_train)와 학습 데이터에 따른 레이블(y_train)을 넘겨서 선형 회귀 모델을 학습시킨다.
model = LogisticRegression(l1_ratio=0.0, C=0.1, solver='saga').fit(x_train, y_train) # 로지스틱 회귀 모델을 만들고 학습시킨다.

# 로지스틱 회귀 계수(가중치)와 상수항(바이어스)
print(model.coef_) # 회귀 계수(가중치)
print(model.intercept_) # 상수항(바이어스)

[[-0.35106496 -0.39952119 -0.34416538 -0.33536206 -0.18281505 -0.03708376
  -0.31270332 -0.41716963 -0.20460366  0.19326562 -0.50771713 -0.00448179
  -0.40934565 -0.36448491  0.10111759  0.25028819  0.09138021 -0.11091941
   0.11185839  0.27028234 -0.4805863  -0.49327914 -0.45309124 -0.42345266
  -0.32649391 -0.16660711 -0.37950955 -0.49676125 -0.37265586 -0.17363876]]
[0.46414288]


학습된 모델로 테스트 데이터를 예측한다.

In [3]:
predict = model.predict(x_test) # predict() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 로지스틱 회귀 모델을 예측한다.
print(predict)

[0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 0 0 0 1 1 0 1 1 0 1 0 1 0 1 0 1 0 1
 0 1 0 0 1 0 1 1 0 1 1 1 0 0 0 0 1 1 1 1 1 1 0 0 0 1 1 0 1 0 0 0 1 1 0 1 0
 0 1 1 1 1 1 0 0 0 1 0 1 1 1 0 0 1 1 0 0 1 1 0 1 1 1 1 1 1 1 0 1 0 1 1 1 1
 0 0 1 1 1 1 1 1 1 1 1 0 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 0 1 1 1 0]


In [4]:
predict_proba = model.predict_proba(x_test) # predict_proba() 메소드의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 각 클래스에 속할 확률로 예측한다.
print(predict_proba)
# 종양은 두 가지(악성, 양성) 클래스로 예측되므로 두 개의 열로 결과가 이루어진 것을 알 수 있다.

[[9.50966326e-01 4.90336739e-02]
 [8.86949564e-02 9.11305044e-01]
 [1.06344505e-02 9.89365550e-01]
 [3.96596415e-02 9.60340358e-01]
 [4.11592987e-03 9.95884070e-01]
 [2.61366007e-02 9.73863399e-01]
 [6.76593605e-03 9.93234064e-01]
 [1.33065107e-02 9.86693489e-01]
 [1.37251174e-03 9.98627488e-01]
 [5.49646979e-04 9.99450353e-01]
 [3.06268979e-01 6.93731021e-01]
 [1.60163263e-01 8.39836737e-01]
 [8.01965856e-04 9.99198034e-01]
 [4.18898390e-01 5.81101610e-01]
 [4.25609370e-01 5.74390630e-01]
 [9.25688885e-01 7.43111146e-02]
 [2.27954881e-02 9.77204512e-01]
 [9.99111450e-01 8.88550271e-04]
 [9.95243323e-01 4.75667719e-03]
 [9.99966456e-01 3.35436858e-05]
 [9.54703771e-01 4.52962285e-02]
 [9.26255445e-01 7.37445549e-02]
 [6.98438795e-02 9.30156120e-01]
 [1.14106224e-02 9.88589378e-01]
 [9.89876818e-01 1.01231817e-02]
 [7.16859712e-03 9.92831403e-01]
 [1.17868351e-03 9.98821316e-01]
 [9.54985267e-01 4.50147334e-02]
 [1.52220050e-02 9.84777995e-01]
 [9.99344550e-01 6.55449716e-04]
 [1.452097

학습된 모델을 평가한다.

In [5]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[49  4]
 [ 1 89]]


In [10]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
# target_names 속성으로 분류 리포트에 실제 레이블의 이름을 출력할 수 있다.
# classification = classification_report(y_test, predict, target_names=['0(악성)', '1(양성)'])
classification = classification_report(y_test, predict, target_names=raw_data.target_names)
print(classification)

              precision    recall  f1-score   support

   malignant       0.98      0.92      0.95        53
      benign       0.96      0.99      0.97        90

    accuracy                           0.97       143
   macro avg       0.97      0.96      0.96       143
weighted avg       0.97      0.97      0.96       143



In [11]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

0.965034965034965


In [12]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[0.98       0.95698925]


In [13]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[0.9245283  0.98888889]


In [15]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[0.95145631 0.9726776 ]
